# Reconstrucción reproducible de df_final
Se validan los 30 archivos esperados antes de calcular características. La ausencia o modificación
de una entrada detiene la ejecución. El movimiento 014 se excluye explícitamente: ver `docs/datos.md`.
Se conserva el CSV publicado y se compara la reconstrucción con tolerancia numérica.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "rehab" / "data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Abre Jupyter desde la raíz del repositorio REHAB.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
from rehab.data import DATA_DIR

from rehab.data import verify_manifest, rebuild, compare_rebuilt
verify_manifest(include_arrays=True)
df_final = rebuild()
print("Diferencia máxima:", compare_rebuilt(df_final))
display(df_final.head())

In [ ]:
from rehab.experiments import save_json, environment
from rehab.data import sha256
output = ROOT / "reports/runs/rebuilt.csv"
output.parent.mkdir(parents=True, exist_ok=True)
if output.exists():
    raise FileExistsError("La salida ya existe; elige otro nombre para conservar su procedencia.")
df_final.to_csv(output, index=False)
save_json(output.with_suffix(".provenance.json"), {
    "generated_sha256": sha256(output), "environment": environment(),
    "input_manifest": verify_manifest(True),
})
print(output)

### Estructura de `df_final`

El DataFrame `df_final` contiene **una fila por cada ventana temporal extraída de una repetición de un movimiento de rehabilitación**. Cada repetición se divide en segmentos de tiempo más pequeños llamados ventanas, y para cada ventana se combina la información de los sensores IMU (`_1`) y del guante (`_2`).

- **`movimiento`**: identifica el ejercicio realizado (`000` a `015`, excluyendo el movimiento `014`).
- **`repeticion_id`**: identifica la repetición del ejercicio dentro de cada archivo.
- **`ventana`**: identifica la ventana de la que fue extraida de cada repetición.
- **Canales de los sensores IMU**: `pitch1`, `yaw1`, `roll1`, `pitch2`, `yaw2` y `roll2`.
- **Canales del guante**: `f1`, `f2`, `f3`, `f4`, `f5` y `pitch3`.

Para cada uno de los 12 canales se calcularon los siguientes estadísticos:

- **`std`** (desviación estándar)
- **`median`** (mediana)
- **`min`** (valor mínimo)
- **`max`** (valor máximo)
- **`iqr`** (rango intercuartílico)
- **`mad_diff`** (desviación absoluta media de las diferencias consecutivas)

Las columnas se nombran combinando el canal y el estadístico. Por ejemplo:

- **`pitch1_std`**: desviación estándar del canal `pitch1`.
- **`f3_median`**: mediana del canal `f3`.
- **`roll2_iqr`**: rango intercuartílico del canal `roll2`.
- **`f5_mad_diff`**: desviación absoluta media de las diferencias consecutivas del canal `f5`.

En total, se generan **72 características numéricas** (`12 canales × 6 estadísticos`), además de las columnas de identificación. La variable que se busca predecir en el problema de clasificación es **`movimiento`**.